# Distributed Hyperparameter Optimization (HPO) Pipeline
## Objective: Flight Delay Classification using cuML & Optuna

This notebook implements a scalable, distributed machine learning pipeline designed for high-performance computing clusters using **Dask-Jobqueue** and **Optuna**.

### Pipeline Architecture
* **Data Processing:** Offloads heavy string parsing and categorical label encoding to CPU nodes using **Polars** for multi-threaded efficiency.
* **Orchestration:** Utilizes **OpOptimizeell API** toayesian hyperparameter optimization.
* **Compute:** Leverages **RAPIDS cuML** (RandomForestClassifier) to train models on L40S GPU nodes, with **RMM (RAPIDS Memory Manager)** utilized to maintain memory stability and prevent fragmentation.
* **Storage:** Data is ingested from **Parquet** files via high-speed shared scratch space to ensure data locality and minimize network bottlenecking.

### Configuration Strategy
1. **Memory Safety:** Every GPU trial initiates an `rmm.reinitialize` call to purge fragmented VRAM pools, ensuring consistent performance across asynchronous iterations.
2. **Concurrency Control:** Limits GPU worker contention by batching trials to match available physical hardware (GPU-per-worker mapping).
3. **Fault Tolerance:** Uses file-path ingestion rather than object scattering to ensure resilience against worker restarts or OOM events.

### Prerequisites
* Ensure the `rapids-23.10` (or compatible) conda/mamba environment is active on the clusteraccess to your own data folderratch/` directory.
* Confirm that GPU workers have the required NVIDIA drivers mapped via `LD_LIBRARY_PATH`.

In [3]:
!pip install kagglehub optuna optuna-integration[dask]

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [1]:
import kagglehub
import os
# Download the "NYC Yellow Taxi Trip Data" variant to a specific directory 
# on your shared cluster filesystem (e.g., your home or scratch directory)
shared_dir = os.path.expanduser("/data/jovillalobos/kaggle")

path = kagglehub.dataset_download(
    "threnjen/2019-airline-delays-and-cancellations",
    output_dir=shared_dir
)

print(f"Dataset downloaded to shared location: {path}")
# Note: Inside this path, look for the CSV name (e.g., 'yellow_tripdata_2016-01.csv')

Dataset downloaded to shared location: /data/jovillalobos/kaggle


In [19]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask_cuda

cluster = SLURMCluster(
    queue='nukwa-l40s',
    cores=1,
    memory='30GB',
    processes=1,
    walltime='02:00:00',
    
    # Crucial: Tells Dask to use the GPU-optimized worker CLI
    worker_command="dask cuda worker", 
    
    # job_extra_directives=["--exclusive"], # Uncomment if you want full node isolation
    
    scheduler_options={
        'port': 8786,                
        'dashboard_address': ':8787' 
    },
    
    job_script_prologue=[
        "module load mamba/3",
        "mamba activate rapids-23.10",
        "export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib64:$LD_LIBRARY_PATH",
    ]
)

# 1. Verify the generated batch script
print("--- Generated SLURM Script ---")
print(cluster.job_script())
print("------------------------------")

# 2. Scale to 2 jobs (submits 2 separate jobs to SLURM, one for each node)
cluster.scale(jobs=2)

# 3. Connect the client to your cluster
client = Client(cluster)
print(f"Dask Dashboard available at: {client.dashboard_link}")

--- Generated SLURM Script ---
#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -p nukwa-l40s
#SBATCH -n 1
#SBATCH --cpus-per-task=1
#SBATCH --mem=28G
#SBATCH -t 02:00:00
module load mamba/3
mamba activate rapids-23.10
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib64:$LD_LIBRARY_PATH
/opt/python/mamba3/envs/rapids-23.10/bin/python -m dask cuda worker tcp://11.0.0.136:8786 --name dummy-name --nthreads 1 --memory-limit 27.94GiB --death-timeout 60

------------------------------
Dask Dashboard available at: http://11.0.0.136:8787/status


[I 2026-06-10 15:24:14,988] A new study created in memory with name: no-name-cb8d59f2-7485-4658-b507-5331d1ec3188
/opt/python/mamba3/envs/rapids-23.10/lib/python3.12/site-packages/distributed/protocol/pickle.py:92: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x)
2026-06-10 15:26:52,848 - distributed.dashboard.components.shared - ERROR - 'NoneType' object has no attribute 'add_next_tick_callback'
Traceback (most recent call last):
  File "/opt/python/mamba3/envs/rapids-23.10/lib/python3.12/site-packages/distributed/utils.py", line 818, in wrapper
    return await func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/mamba3/envs/rapids-23.10/lib/python3.12/site-packages/distributed/dashboard/components/shared.py", line 327, in cb
    self.doc().add_next_tick_callback(lambda: self.update(prof, metadata))
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' obje

In [29]:
from dask.distributed import Client

# Connect to your existing cluster
client = Client('tcp://11.0.0.136:8786')
client

<Client: 'tcp://11.0.0.136:8786' processes=2 threads=2, memory=55.88 GiB>

In [30]:
import dask.dataframe as dd

# Define your paths

train_path = os.path.join(path, "train.csv")
test_path = os.path.join(path, "test.csv")

parquet_train = os.path.join(path, "train.parquet")
parquet_test = os.path.join(path, "test.parquet")

if not (os.path.exists(parquet_train) or os.path.exists(parquet_test)): 
    
    print("Converting Train CSV to Parquet...")
    # Dask reads the CSV in chunks and streams it to compressed Parquet format
    df_train = dd.read_csv(train_path, assume_missing=True)
    df_train.to_parquet(parquet_train, engine='pyarrow', compression='snappy')
    
    print("Converting Test CSV to Parquet...")
    df_test = dd.read_csv(test_path, assume_missing=True)
    df_test.to_parquet(parquet_test, engine='pyarrow', compression='snappy')
    
    print("Conversion Complete!")

else: 
    print("Already converted!")

Already converted!


In [31]:
def gpu_test():
    from numba import cuda
    import socket

    return {
        "host": socket.gethostname(),
        "cuda": cuda.is_available(),
        "gpu": str(cuda.get_current_device().name),
    }

import pprint
pprint.pprint(client.run(gpu_test))

{'tcp://11.0.0.111:37609': {'cuda': True,
                            'gpu': "b'NVIDIA L40S'",
                            'host': 'nukwa-06.cnca'},
 'tcp://11.0.0.111:38099': {'cuda': True,
                            'gpu': "b'NVIDIA L40S'",
                            'host': 'nukwa-06.cnca'}}


In [32]:
def remote_objective(trial, train_path, test_path):
    import gc
    import rmm
    import polars as pl
    import cudf
    from cuml.ensemble import RandomForestClassifier
    from cuml.metrics import accuracy_score
    # 1. Reset the GPU Memory Pool
    if rmm.is_initialized():
        rmm.reinitialize(managed_memory=True)
    else:
        rmm.initialize(managed_memory=True)


    # ==========================================
    # 2. CPU Data Processing (Polars)
    # ==========================================
    target_col = "DEP_DEL15"
    cat_cols = [
        "DEP_TIME_BLK",
        "CARRIER_NAME",
        "DEPARTING_AIRPORT",
        "PREVIOUS_AIRPORT",
    ]

    # Load data
    train_pl = pl.read_parquet(train_path).drop_nulls()
    test_pl = pl.read_parquet(test_path).drop_nulls()

    # Consistent categorical encoding
    for col in cat_cols:
        if col in train_pl.columns:
            unique_values = (
                pl.concat([
                    train_pl.select(col),
                    test_pl.select(col)
                ])
                .unique()
                .with_row_index("code")
            )

            train_pl = (
                train_pl
                .join(unique_values, on=col)
                .drop(col)
                .rename({"code": col})
            )

            test_pl = (
                test_pl
                .join(unique_values, on=col)
                .drop(col)
                .rename({"code": col})
            )

    features = [c for c in train_pl.columns if c != target_col]

    # Cast to float32/int32 during Pandas conversion for GPU efficiency
    X_train_pd = train_pl.select(features).to_pandas().astype('float32')
    y_train_pd = train_pl[target_col].to_pandas().astype('int32')
    X_test_pd = test_pl.select(features).to_pandas().astype('float32')
    y_test_pd = test_pl[target_col].to_pandas().astype('int32')

    # Aggressive CPU cleanup
    del train_pl, test_pl
    gc.collect()

    # ==========================================
    # 3. GPU Handoff & Training (RAPIDS)
    # ==========================================
    
    # Load into L40S VRAM
    X_train = cudf.DataFrame.from_pandas(X_train_pd)
    y_train = cudf.Series.from_pandas(y_train_pd)
    X_test = cudf.DataFrame.from_pandas(X_test_pd)
    y_test = cudf.Series.from_pandas(y_test_pd)

    # Free up host node memory
    del X_train_pd, y_train_pd, X_test_pd, y_test_pd
    gc.collect()

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=25),
        "max_depth": trial.suggest_int("max_depth", 6, 16),
        "max_features": trial.suggest_float("max_features", 0.5, 0.9),
    }
    
    # Train Model
    model = RandomForestClassifier(
        **params,
        n_bins=128,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)
    acc = float(accuracy_score(y_test, preds))

    # Final sweep of GPU memory before returning to Optuna
    del X_train, y_train, X_test, y_test, model, preds
    gc.collect()

    return acc

In [ ]:
# Because these are strings, Dask will serialize this partial perfectly!
import optuna 
import optuna_integration
from functools import partial

objective_partial = partial(
    remote_objective,
    train_path=parquet_train,
    test_path=parquet_test
)

# ==========================================
# 3. RUN DISTRIBUTED OPTIMIZATION
# ==========================================
storage = optuna_integration.DaskStorage()

study = optuna.create_study(
    direction="maximize",
    storage=storage,
)

print("Submitting distributed study.optimize trials...")

# Your exact original submission logic
futures = [
    client.submit(
        study.optimize,
        objective_partial,
        n_trials=1,
        pure=False,
    )
    for _ in range(15)
]

# Wait for the cluster to finish all 15 trials
client.gather(futures)

In [36]:
print("\nOptimization completed")
print("Best score:", study.best_value)
print("Best parameters:")

for k, v in study.best_params.items():
    print(f"{k}: {v}")


Optimization completed
Best score: 0.824011066825772
Best parameters:
n_estimators: 250
max_depth: 16
max_features: 0.867861482811712


In [43]:
import polars as pl
import os

# Update this to your actual shared path

print("Inflating Parquet into system RAM...")

# 1. Scan (Lazy) and Collect (Materialize)
# This forces Polars to load the raw data into RAM
df = pl.scan_parquet(parquet_train).collect()

# 2. Calculate Memory Usage
# estimated_size() gives the total bytes occupied by the DataFrame in memory
total_bytes = df.estimated_size()
total_gb = total_bytes / (1024**3) # Converting to GiB

print(f"\n--- Memory Report ---")
print(f"Number of rows: {len(df):,}")
print(f"Total RAM inflated size: {total_gb:.2f} GiB")

# 2. Correct way to get column-level breakdown
# We iterate over the columns and get the size of each Series
mem_data = {
    "column": df.columns,
    "bytes": [df[c].estimated_size() for c in df.columns]
}

mem_breakdown = pl.DataFrame(mem_data)

print("\n--- Top 5 Memory-Consuming Columns ---")
print(mem_breakdown.sort("bytes", descending=True).head(5))
del df

Inflating Parquet into system RAM...

--- Memory Report ---
Number of rows: 4,542,343
Total RAM inflated size: 1.23 GiB

--- Top 5 Memory-Consuming Columns ---
shape: (5, 2)
┌───────────────────┬───────────┐
│ column            ┆ bytes     │
│ ---               ┆ ---       │
│ str               ┆ i64       │
╞═══════════════════╪═══════════╡
│ DEPARTING_AIRPORT ┆ 111088803 │
│ CARRIER_NAME      ┆ 93271850  │
│ PREVIOUS_AIRPORT  ┆ 89151649  │
│ DEP_TIME_BLK      ┆ 40881087  │
│ MONTH             ┆ 36338744  │
└───────────────────┴───────────┘


In [46]:
import warnings
# Silence the GPU detection warning from cudf on the CPU-only head node
warnings.filterwarnings("ignore", category=UserWarning, message="No NVIDIA GPU detected")

import cudf
import dask_cudf
# Now you can import without the annoying warning
# This runs 'nvidia-smi' on all your workers and returns the results to your head node
print(client.run(lambda: __import__('subprocess').check_output(['nvidia-smi', '-L']).decode()))

{'tcp://11.0.0.111:37609': 'GPU 0: NVIDIA L40S (UUID: GPU-c7a1349b-a7ec-70ea-622f-546d08b78f99)\n', 'tcp://11.0.0.111:38099': 'GPU 0: NVIDIA L40S (UUID: GPU-c7a1349b-a7ec-70ea-622f-546d08b78f99)\n'}


In [48]:
# Check environment variables on the workers
print(client.run(lambda: __import__('os').environ.get('LD_LIBRARY_PATH')))

{'tcp://11.0.0.111:37609': '/usr/local/cuda/lib64:/usr/lib64:/opt/python/mamba3/lib:/usr/local/cuda/lib64', 'tcp://11.0.0.111:38099': '/usr/local/cuda/lib64:/usr/lib64:/opt/python/mamba3/lib:/usr/local/cuda/lib64'}


In [57]:
import glob

# 1. Get a list of all your parquet files on the shared drive
files = glob.glob(os.path.join(path,"/*.parquet"))

print(files)

def process_single_file(file_path):
    """
    This function runs entirely inside the GPU worker's environment.
    It imports its own libraries and doesn't communicate with the head node.
    """
    import cudf
    import os
    
    # Process only this one file
    df = cudf.read_parquet(file_path)
    df['IS_RUSH_HOUR'] = df['DEP_TIME_BLK'].str.contains('1500|1600|1700|1800').astype('int32')
    
    # Save to a 'processed' folder
    out_dir = os.path.expanduser("~/shared_scratch/large_dataset/processed/")
    os.makedirs(out_dir, exist_ok=True)
    
    output_path = os.path.join(out_dir, os.path.basename(file_path))
    df.to_parquet(output_path)
    
    return f"Processed {file_path}"

# 2. Submit jobs to the cluster
# We use client.map to distribute the file list across available workers
futures = client.map(process_single_file, files)

# 3. Gather results
results = client.gather(futures)
print(results)

[]
[]


In [58]:
!ls /tmp

casch
dask-scratch-space
hsperfdata_luarrieta
hsperfdata_taller-3436
ompi.kura-1c.15161
systemd-private-877abbe68d6747fe80bb406c9fbc7180-chronyd.service-CrroUc
tmp0ygb19j7
tmp.19DBSf9mBl
tmp.1FNQPqOQra
tmp2aqv7xbr
tmp2gm38c4n
tmp30v4jydt
tmp41qgu6mn
tmp4e2paa5a
tmp5dasv9kz
tmp5o8e3f_s
tmp6nwlu2bi
tmp.6zYRpQopqS
tmp.7JCUi90x8r
tmp7jt89yzc
tmp9fzdlcz1
tmpbkgvoj9l
tmpcbflsjia
tmpcjkiwv96
tmpcoiftw3a
tmp.D5AkwrOHH2
tmpdigf52i9
tmp.eDKuFcTBvj
tmpehbpyfk5
tmp.fWcVx0zpjG
tmpgcjoqxs5
tmpgg_06m0h
tmp.GtgUtZq4IR
tmphpb_rlym
tmp.IckLZLi3mD
tmpik1q6xq0
tmp.iNSV6MFtNU
tmpj75ip9cj
tmp.jMsUnk2ZML
tmpjz15y_lw
tmp.K6oDkjMKBD
tmp.LkKEJW9RR8
tmplp6wriyi
tmpm58z0fjs
tmp.mhvA2tP0sz
tmpmr1r1_tq
tmpnsjqlh6w
tmpo8z85m6a
tmp.pLgxuWMdnS
tmp.pyVfmrKAQN
tmp.qbhwPvBSjF
tmp.qeAHj8ofpg
tmp.QuB51KcGrc
tmprjhlo2w9
tmp.s7kqCStA7j
tmpsex_vhis
tmp.T2esCNTY5Q
tmpt2h0318l
tmp.tceA9p6z4x
tmptxvsnmge
tmp.ulvmjlAwW6
tmpuxzazb22
tmp.vPuCsgoCDV
tmpwrl9cwys
tmpxclxcbpp
tmpz0iv9nfd
tmp.ZqTJcKtPEj
tmp.zrEIta4Hgn


In [ ]:
client.close()
cluster.close()